Q1

In [ ]:
#This script is run inside main.py

import os
import logfire
from dotenv import load_dotenv

load_dotenv()

logfire.configure()
logfire.instrument_pydantic_ai()

from agent import faq_agent, SearchDeps
from ingest import build_index, load_faq_data


def main():

    documents = load_faq_data()
    index = build_index(documents)

    deps = SearchDeps(index=index)

    question = 'How do I run Ollama locally?'
    result = faq_agent.run_sync(question, deps=deps)

    print(result.output)


if __name__ == '__main__':
    main()

#The answer is 5

Q2

In [ ]:
#This script is run inside python logfire_pipeline.py

from dotenv import load_dotenv
load_dotenv()

import dlt
from dlt.sources.rest_api import rest_api_source

def logfire_traces_source():
    return rest_api_source({
        "client": {
            "base_url": "https://logfire-api.pydantic.dev/v1/",
            "auth": {
                "type": "bearer",
                "token": dlt.secrets["LOGFIRE_READ_TOKEN"]
            }
        },
        "resources": [
            {
                "name": "records",
                "endpoint": {
                    "path": "query",
                    "params": {
                        "sql": "SELECT * FROM records"
                    },
                    "data_selector": "data"
                }
            }
        ]
    })

def load_logfire() -> None:
    pipeline = dlt.pipeline(
        pipeline_name="logfire_pipeline",
        destination="duckdb",
        dataset_name="agent_traces",
    )

    load_info = pipeline.run(logfire_traces_source())
    print(load_info)

if __name__ == "__main__":
    load_logfire()



Q3
The input tokens from the previously generated query is displayed in the logfire dashboard. The answer is exactly 1710 tokens.